In [5]:
import commons as c
import plotly.graph_objects as go
import plotly.express as px

In [6]:
def cap_value(value):
    return min(1, max(0, value))

def get_tolerance_values_noisy(model, threshold):
    # Define tolerance values
    if threshold == 'I':
        tolerance_values_noisy = c.tolerance_values_ideal
    elif threshold == 'N':
        tolerance_values_noisy = c.getModelTolerance(model)
    elif threshold == 'A':
        tolerance_values_noisy = c.getModelTolerance(model)
        tolerance_values_noisy = {
            'fidelity':  cap_value(1 - ((1 - tolerance_values_noisy['fidelity']) + (1 - c.tolerance_values_ideal['fidelity']))),
            'trace': cap_value(tolerance_values_noisy['trace'] + c.tolerance_values_ideal['trace']),
            'hellinger': cap_value(tolerance_values_noisy['hellinger'] + c.tolerance_values_ideal['hellinger']),
            'jensenshannon': cap_value(tolerance_values_noisy['jensenshannon'] + c.tolerance_values_ideal['jensenshannon']),
            'chisquare': cap_value(tolerance_values_noisy['chisquare'] + c.tolerance_values_ideal['chisquare']),
            'expectation': cap_value(tolerance_values_noisy['expectation'] + c.tolerance_values_ideal['expectation'])
        }

    else:
        tolerance_values_noisy = {
            'fidelity': 1 - threshold,
            'trace': threshold,
            'hellinger': threshold,
            'jensenshannon': threshold,
            'chisquare': threshold,
            'expectation': threshold
        }

    return tolerance_values_noisy


In [7]:
def get_grouped_bar_chart(noise_model):
    fig = go.Figure()
    color_scale = px.colors.qualitative.Bold

    # Add bars for each category
    for i, threshold in enumerate(c.thresholds):  # Assuming c.thresholds is a list like [0.1, 0.5, 0.8, 'A', 'I', 'N']
        tolerance_values_noisy = get_tolerance_values_noisy(noise_model, threshold)
        y_values = [tolerance_values_noisy[metric] for metric in c.metrics.values()]
        fig.add_trace(go.Bar(
            name=str(threshold),  # Ensure threshold is converted to string for labels
            x=list(c.metrics.keys()),  # Use keys from the dictionary for x-axis
            y=y_values,  # F1 scores
            hoverinfo='y',
            marker=dict(color=color_scale[i % len(color_scale)])
        ))

    # Add horizontal lines for the ideal tolerance values
    for metric_index, (key, metric) in enumerate(c.metrics.items()):  # Iterate over the dictionary
        ideal_value = c.tolerance_values_ideal.get(metric)
        fig.add_shape(
            type="line",
            x0=metric_index - 0.5,  # Start of the group
            x1=metric_index + 0.5,  # End of the group
            y0=ideal_value, y1=ideal_value, xref='x',
            yref='y',
            line=dict(color="red", width=3)
        )

    return fig


In [8]:
for hw in c.hardware:
    fig = get_grouped_bar_chart(hw)
    folder = "results/thresholds_analyse/"
    c.setup_layout_and_save(fig, f"Thresholds comparison for {hw} noise model", folder, f"thresholds_{hw}")
    fig.show()